In [1]:
import pandas as pd

In [2]:
def encode_soil_data(df, soil_col="DOMSOI", min_count=20, drop_original=True):
    """
    One-hot encodes the soil classification column.
 
    Parameters:
        df           : your dataframe
        soil_col     : which column to encode (default "DOMSOI")
        min_count    : categories occurring fewer times than this get
                       grouped into "Other" (default 20)
        drop_original: if True, removes the original text column after
                       encoding (recommended - model can't use raw text anyway)
 
    Returns a new dataframe with additional DOMSOI_<category> columns.
    """
    df = df.copy()
 
    # Step 1: Group rare categories into "Other"
    value_counts = df[soil_col].value_counts()
    rare_categories = value_counts[value_counts < min_count].index.tolist()
 
    print(f"Total unique '{soil_col}' categories: {df[soil_col].nunique()}")
    print(f"Categories below min_count={min_count} (grouped into 'Other'): {rare_categories}")
 
    df[f"{soil_col}_grouped"] = df[soil_col].apply(
        lambda x: "Other" if x in rare_categories else x
    )
 
    # Step 2: One-hot encode the grouped column
    dummies = pd.get_dummies(
        df[f"{soil_col}_grouped"],
        prefix=soil_col,
        dtype=int,  # ensures 0/1 integers, not True/False
    )
 
    df = pd.concat([df, dummies], axis=1)
 
    # Step 3: Clean up intermediate/original columns
    df = df.drop(columns=[f"{soil_col}_grouped"])
    if drop_original:
        df = df.drop(columns=[soil_col])
 
    print(f"\nFinal soil-related columns created: {list(dummies.columns)}")
    print(f"New shape: {df.shape}")
 
    return df

In [4]:
df = pd.read_csv("NER_landslide_combined_final.csv")

In [5]:
df = encode_soil_data(df)

Total unique 'DOMSOI' categories: 14
Categories below min_count=20 (grouped into 'Other'): ['Gh', 'Bf', 'Je', 'Ah']

Final soil-related columns created: ['DOMSOI_Af', 'DOMSOI_Ao', 'DOMSOI_Bd', 'DOMSOI_Be', 'DOMSOI_Bh', 'DOMSOI_Gd', 'DOMSOI_Ge', 'DOMSOI_I', 'DOMSOI_Nd', 'DOMSOI_Other', 'DOMSOI_Rd']
New shape: (1398, 30)


In [6]:
df.to_csv("NER_landslide_combined_final.csv", index=False)